In [14]:
import torch
import sys, os

project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from peptide_pipeline.generator.cvae_generator_agent.cvae_generator import CVAEGenerator


In [15]:
repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
sys.path.insert(0, repo_root)

# Import the JSONDataLoader class from the dataloader_json.py in dataloader module
from peptide_pipeline.dataloader.dataloader_json import DataLoader as JSONDataLoader

json_loader = JSONDataLoader()
json_loader.load_data("database/training_data.json", columns=["sequence","length","cathionicity","hydrophobicity"])
print(f"Loaded {len(json_loader.data)} peptides from JSON file")
data = json_loader.get_data()

filtered_data = data[(data["length"] >= 5) & (data["length"] <= 14)]
print(f"Filtered peptides (length 5-14): {len(filtered_data)}")

properties = ["length", "cathionicity", "hydrophobicity"]
stats = {}
for prop in properties:
    min_val = float(filtered_data[prop].min())
    max_val = float(filtered_data[prop].max())
    mean_val = float(filtered_data[prop].mean())
    stats[prop] = {"min": min_val, "max": max_val, "mean": mean_val}
    print(f"{prop}: min={min_val:.4f}, max={max_val:.4f}, mean={mean_val:.4f}")

# Scalar conditioning values (baseline-style)
constraints_default = {
    "size": stats["length"]["mean"],
    "molecular_weight": 1500.0,
    "net_charge_pH5_5": 0.0,
    "isoelectric_point": 7.0,
    "hydrophobicity": stats["hydrophobicity"]["mean"],
    "cathionicity": stats["cathionicity"]["mean"],
    "hydrophobic_moment": 0.5,
    "logp": 0.0,
    "boman_index": 0.0,
    "h_bond_donors": 5.0,
    "h_bond_acceptors": 5.0,
    "tpsa": 100.0,
}

#  --- Initialize model ---
cvae = CVAEGenerator(latent_dim=16, hidden_dim=64, condition_dim=32)
print(f"Device:     {cvae.device}")
print(f"Input dim:  {cvae.input_dim}")
print(f"Latent dim: {cvae.latent_dim}")
print(f"Hidden dim: {cvae.hidden_dim}")
print(f"Condition dim: {cvae.condition_dim}")

conditions = cvae._constraints_to_condition_tensor(
    constraints=constraints_default,
    count=1,
    device=cvae.device,
)
print(f"Condition tensor shape: {conditions.shape}")

MAX_LEN = 14
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
PAD_IDX = 20
VOCAB_SIZE = 21

sequences = [s for s in filtered_data["sequence"].tolist() if 5 <= len(s) <= MAX_LEN]
lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long, device=cvae.device)

def encode_with_pad(seqs, max_len):
    x = torch.zeros(len(seqs), max_len * VOCAB_SIZE)
    for i, seq in enumerate(seqs):
        for j in range(max_len):
            x[i, j * VOCAB_SIZE + PAD_IDX] = 1.0
        for j, aa in enumerate(seq[:max_len]):
            if aa in aa_index:
                x[i, j * VOCAB_SIZE + PAD_IDX] = 0.0
                x[i, j * VOCAB_SIZE + aa_index[aa]] = 1.0
    return x

x = encode_with_pad(sequences, MAX_LEN).to(cvae.device)

conditions = cvae._constraints_to_condition_tensor(
    constraints=constraints_default,
    count=len(sequences),
    device=cvae.device
)

cvae = CVAEGenerator(max_len=MAX_LEN, latent_dim=16, hidden_dim=64, condition_dim=32)
cvae.train_model(data=x, conditions=conditions, lengths=lengths, epochs=500, batch_size=64, lr=1e-3, kl_anneal_epochs=100)

Loaded 3653 peptides from JSON file
Filtered peptides (length 5-14): 3210
length: min=5.0000, max=12.0000, mean=9.7259
cathionicity: min=0.0000, max=12.0000, mean=3.5237
hydrophobicity: min=-3.8900, max=4.5000, mean=0.5256
Device:     cuda
Input dim:  294
Latent dim: 16
Hidden dim: 64
Condition dim: 32
Condition tensor shape: torch.Size([1, 32])


In [16]:
# --- Metrics helpers ---
training_peptides_set = set(sequences)
amino_acid_set = set(AMINO_ACIDS)

def evaluate_generated_peptides(generated_peptides, training_set, amino_set, expected_len=None):
    total = len(generated_peptides)
    if total == 0:
        return {
            "count": 0,
            "validity": 0.0,
            "diversity": 0.0,
            "novelty_vs_training": 0.0,
            "unique_count": 0,
            "new_count": 0,
        }

    def is_valid(peptide):
        chars_ok = all(aa in amino_set for aa in peptide)
        len_ok = True if expected_len is None else len(peptide) == expected_len
        return chars_ok and len_ok

    valid_count = sum(1 for peptide in generated_peptides if is_valid(peptide))
    unique_generated = set(generated_peptides)
    new_peptides = [peptide for peptide in unique_generated if peptide not in training_set]

    return {
        "count": total,
        "validity": valid_count / total,
        "diversity": len(unique_generated) / total,
        "novelty_vs_training": len(new_peptides) / len(unique_generated) if unique_generated else 0.0,
        "unique_count": len(unique_generated),
        "new_count": len(new_peptides),
    }

def print_metrics(label, metrics):
    print(f"[{label}] count={metrics['count']}")
    print(f"[{label}] validity={metrics['validity']:.2%}")
    print(f"[{label}] diversity={metrics['diversity']:.2%}")
    print(f"[{label}] novelty_vs_training={metrics['novelty_vs_training']:.2%} ({metrics['new_count']}/{metrics['unique_count']})")

In [17]:
# --- Generate ---
generated = cvae.generate_peptides(count=10, constraints=constraints_default, temperature=0.8)
expected_len = max(1, min(MAX_LEN, int(round(constraints_default["size"]))))

for i, pep in enumerate(generated):
    valid = all(c in amino_acid_set for c in pep) and len(pep) == expected_len
    status = "OK" if valid else "INVALID"
    print(f"Peptide {i+1:02d}: {pep}  {status}")

metrics = evaluate_generated_peptides(
    generated_peptides=generated,
    training_set=training_peptides_set,
    amino_set=amino_acid_set,
    expected_len=expected_len,
)
print_metrics("default_constraints", metrics)

Peptide 01: RLLRKLKRWW  OK
Peptide 02: GHFWRAILWL  OK
Peptide 03: CLIRKRLWKR  OK
Peptide 04: WWKWVWWRLI  OK
Peptide 05: KRLGFLISKR  OK
Peptide 06: RKRRKGVKRR  OK
Peptide 07: LWRRAMELFK  OK
Peptide 08: LVWRIKSKRK  OK
Peptide 09: GAFLSKRWKL  OK
Peptide 10: RLKARFVKRL  OK
[default_constraints] count=10
[default_constraints] validity=100.00%
[default_constraints] diversity=100.00%
[default_constraints] novelty_vs_training=100.00% (10/10)


In [18]:
# Test with direct scalar constraints and check if generated peptides meet those constraints
constraints = {
    "size": 12,
    "molecular_weight": 1500.0,
    "net_charge_pH5_5": 4.0,
    "isoelectric_point": 10.0,
    "hydrophobicity": 0.10,
    "cathionicity": stats["cathionicity"]["mean"],
    "hydrophobic_moment": 0.5,
    "logp": 0.0,
    "boman_index": 0.0,
    "h_bond_donors": 5.0,
    "h_bond_acceptors": 5.0,
    "tpsa": 100.0,
}

generated = cvae.generate_peptides(
    count=10,
    constraints=constraints,
    temperature=0.8
)

expected_len = int(round(constraints["size"]))
for i, pep in enumerate(generated, 1):
    status = "OK" if len(pep) == expected_len and all(c in amino_acid_set for c in pep) else "INVALID"
    print(f"{i:02d}. {pep}  {status}")

metrics = evaluate_generated_peptides(
    generated_peptides=generated,
    training_set=training_peptides_set,
    amino_set=amino_acid_set,
    expected_len=expected_len,
)
print_metrics("custom_constraints", metrics)

01. GWLQLWRIWFKL  OK
02. GLRCIKVKKRRK  OK
03. KLKRWWVKKCRK  OK
04. GLVIWRKGKWVK  OK
05. KRIWWLRLKVRL  OK
06. WRKAWLLLCRKR  OK
07. GKLLIKRKWFKL  OK
08. KWWRLLLVAKRL  OK
09. GWLFIRKLVLLK  OK
10. VLWWAIVRWWRK  OK
[custom_constraints] count=10
[custom_constraints] validity=100.00%
[custom_constraints] diversity=100.00%
[custom_constraints] novelty_vs_training=100.00% (10/10)
